# 🧠 Adaptive Query-Routing Agentic System v3
## Hybrid: Local Domain Expert + Multi-Provider API Routing

**Model Lineup — 4 models, 3 organizations, hybrid local+API:**

| Role | Model | Org | Source | Why |
|------|-------|-----|--------|-----|
| Router + Simple QA | LLaMA 3.1 8B | Meta | Groq API | Fast, cheap for easy Qs |
| Medical Agent | **BioMistral-7B** | **EPFL/Mistral** | **Local GPU (4-bit)** | **Trained on 48B tokens of PubMed** |
| Math Agent | Qwen3 32B | Alibaba | Groq API | Strong reasoning & math |
| Synthesizer | LLaMA 3.3 70B | Meta | Groq API | Large generalist |

**Why this architecture:**
- BioMistral is the only model actually *trained* on medical literature — not just prompted
- Qwen3 32B is a reasoning model designed for complex step-by-step problems
- LLaMA 8B handles simple questions without wasting expensive model calls
- LLaMA 70B synthesizes multi-expert answers coherently

**Requires:** Colab GPU (T4) for BioMistral + Groq API key (free)

In [1]:
!pip install -q openai transformers accelerate bitsandbytes sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00


In [ ]:
import os, json, re, time, gc, torch
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# ═══════════════════════════════════════════════════════════
# GROQ API SETUP
# ═══════════════════════════════════════════════════════════
GROQ_API_KEY = ""  # ← Paste your key
if not GROQ_API_KEY:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except: pass
assert GROQ_API_KEY, "Need Groq API key! Free at console.groq.com/keys"

groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)

# ═══════════════════════════════════════════════════════════
# LOAD BIOMISTRAL LOCALLY (4-bit, ~4GB VRAM)
# ═══════════════════════════════════════════════════════════
print("Loading BioMistral-7B (4-bit quantized)...")
BNB_CONFIG = BitsAndBytesConfig(
  load_in_4bit=True, bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
)
bio_tokenizer = AutoTokenizer.from_pretrained("BioMistral/BioMistral-7B", trust_remote_code=True)
if bio_tokenizer.pad_token is None:
  bio_tokenizer.pad_token = bio_tokenizer.eos_token
bio_model = AutoModelForCausalLM.from_pretrained(
  "BioMistral/BioMistral-7B", quantization_config=BNB_CONFIG,
  device_map="auto", trust_remote_code=True,
)
print(f"  ✅ BioMistral loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Verify Groq models
API_MODELS = {
    "router":      "llama-3.1-8b-instant",
    "math":        "qwen/qwen3-32b",
    "synthesizer": "llama-3.3-70b-versatile",
}
for role, mid in API_MODELS.items():
    try:
        groq_client.chat.completions.create(
            model=mid, messages=[{"role":"user","content":"Say ok"}], max_tokens=5, temperature=0)
        print(f"  ✅ {role:12s} → {mid}")
    except Exception as e:
        print(f"  ❌ {role:12s} → {mid}: {e}")

print("\n✅ All models ready.")

Loading BioMistral-7B (4-bit quantized)...


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 47, in spawn_con

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

  ✅ BioMistral loaded. VRAM: 4.1 GB
  ✅ router       → llama-3.1-8b-instant
  ✅ math         → qwen/qwen3-32b
  ✅ synthesizer  → llama-3.3-70b-versatile

✅ All models ready.


In [6]:
# ═══════════════════════════════════════════════════════════
# UNIFIED LLM INTERFACE + COST TRACKING
# ═══════════════════════════════════════════════════════════

COST_PER_M = {
    "llama-3.1-8b-instant":     {"in": 0.05, "out": 0.08},
    "qwen/qwen3-32b":          {"in": 0.20, "out": 0.60},
    "llama-3.3-70b-versatile":  {"in": 0.60, "out": 0.89},
    "biomistral-local":         {"in": 0.00, "out": 0.00},  # free (local GPU)
}

_call_log = []

def call_api(prompt, system="", model="llama-3.3-70b-versatile", max_tokens=512, temperature=0.1):
    """Call a Groq API model."""
    messages = []
    if system: messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    t0 = time.time()
    try:
        resp = groq_client.chat.completions.create(
            model=model, messages=messages, max_tokens=max_tokens, temperature=temperature)
        elapsed = time.time() - t0
        text = resp.choices[0].message.content.strip()
        if '<think>' in text:
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        usage = resp.usage
        costs = COST_PER_M.get(model, {"in":0.5,"out":0.5})
        cost = usage.prompt_tokens*costs["in"]/1e6 + usage.completion_tokens*costs["out"]/1e6
        meta = {"model":model,"latency":elapsed,"input_tokens":usage.prompt_tokens,
                "output_tokens":usage.completion_tokens,"total_tokens":usage.total_tokens,"cost":cost}
        _call_log.append(meta)
        return text, meta
    except Exception as e:
        if "429" in str(e):
            print(f"  ⏳ Rate limited. Waiting 15s...")
            time.sleep(15)
            return call_api(prompt, system, model, max_tokens, temperature)
        print(f"  ⚠ API error ({model}): {e}")
        return "", {"model":model,"latency":time.time()-t0,"cost":0,"total_tokens":0}


def call_biomistral(prompt, system="", max_tokens=300):
    """Call the locally loaded BioMistral model."""
    if system:
        input_text = f"{system}\n\n{prompt}\n\nAnswer:"
    else:
        input_text = f"{prompt}\n\nAnswer:"
    inputs = bio_tokenizer(input_text, return_tensors="pt").to(bio_model.device)
    t0 = time.time()
    with torch.no_grad():
        outputs = bio_model.generate(
            **inputs, max_new_tokens=max_tokens, temperature=0.1,
            do_sample=True, pad_token_id=bio_tokenizer.eos_token_id)
    elapsed = time.time() - t0
    text = bio_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    in_tok = inputs['input_ids'].shape[1]
    out_tok = outputs.shape[1] - in_tok
    meta = {"model":"biomistral-local","latency":elapsed,"input_tokens":in_tok,
            "output_tokens":out_tok,"total_tokens":in_tok+out_tok,"cost":0.0}
    _call_log.append(meta)
    return text, meta

print("Unified LLM interface ready (API + local).")

Unified LLM interface ready (API + local).


In [7]:
# ═══════════════════════════════════════════════════════════
# AGENTS
# ═══════════════════════════════════════════════════════════

@dataclass
class SubQuestion:
    text: str; domain: str; answer: str = ""; model_used: str = ""
    latency: float = 0.0; tokens: int = 0; cost: float = 0.0

@dataclass
class QueryTrace:
    query: str; classification: str = ""; reasoning: str = ""
    sub_questions: List[SubQuestion] = field(default_factory=list)
    final_answer: str = ""; total_time: float = 0.0
    total_tokens: int = 0; total_cost: float = 0.0
    models_used: List[str] = field(default_factory=list)

MEDICAL_SYSTEM = (
    "You are a biomedical expert. Answer the following medical question "
    "accurately and concisely based on your clinical and pharmacological knowledge."
)

CLASSIFIER_SYSTEM = (
    "You are a query classifier. Respond with ONLY valid JSON, no markdown, no backticks.")
CLASSIFIER_PROMPT = """Classify this question:
- SIMPLE: answerable in one step from general knowledge
- MULTI_HOP: needs multiple reasoning steps or domain expertise

If MULTI_HOP, decompose into 2-4 sub-questions tagged with: medical, math, or general

JSON format:
Simple: {"type": "simple", "reasoning": "why"}
Multi-hop: {"type": "multi_hop", "reasoning": "why", "sub_questions": [{"q": "text", "domain": "medical"}]}

Question: """

def classify_query(query):
    raw, meta = call_api(CLASSIFIER_PROMPT + query, system=CLASSIFIER_SYSTEM,
                         model=API_MODELS["router"], max_tokens=400, temperature=0.1)
    try:
        parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
    except:
        return "simple", "parse failed", [], meta
    q_type = parsed.get("type", "simple").lower().strip()
    subs = []
    if q_type == "multi_hop":
        for sq in parsed.get("sub_questions", []):
            d = sq.get("domain", "general").lower()
            if d in ("biomedical","bio","medicine","health","clinical"): d = "medical"
            elif d in ("mathematics","calculation","statistics"): d = "math"
            elif d not in ("medical","math"): d = "general"
            subs.append(SubQuestion(text=sq["q"], domain=d))
    return q_type, parsed.get("reasoning",""), subs, meta


def answer_sub_question(sq):
    """Route to the right model based on domain."""
    if sq.domain == "medical":
        # BioMistral — LOCAL, domain-trained on PubMed
        answer, meta = call_biomistral(sq.text, system=MEDICAL_SYSTEM, max_tokens=300)
    elif sq.domain == "math":
        # Qwen3 32B — API, reasoning model
        system = "Show your work step by step. Double-check arithmetic. Present final answer clearly."
        answer, meta = call_api(sq.text, system=system, model=API_MODELS["math"], max_tokens=400)
    else:
        answer, meta = call_api(sq.text, model=API_MODELS["synthesizer"], max_tokens=300)
    sq.answer = answer
    sq.model_used = meta.get("model", "unknown")
    sq.latency = meta.get("latency", 0)
    sq.tokens = meta.get("total_tokens", 0)
    sq.cost = meta.get("cost", 0)
    return sq


def answer_parallel(subs):
    # NOTE: BioMistral calls are sequential (GPU), API calls are parallel
    medical_subs = [(i,sq) for i,sq in enumerate(subs) if sq.domain == "medical"]
    other_subs = [(i,sq) for i,sq in enumerate(subs) if sq.domain != "medical"]
    results = [None]*len(subs)
    # Run medical (local) sequentially
    for i, sq in medical_subs:
        results[i] = answer_sub_question(sq)
    # Run others (API) in parallel
    if other_subs:
        with ThreadPoolExecutor(max_workers=len(other_subs)) as ex:
            futs = {ex.submit(answer_sub_question, sq): i for i, sq in other_subs}
            for f in as_completed(futs):
                results[futs[f]] = f.result()
    return results


def synthesize(query, subs):
    expert_text = ""
    for i, sq in enumerate(subs, 1):
        expert_text += f"\n[{sq.domain.upper()} — {sq.model_used}] Q: {sq.text}\nA: {sq.answer}\n"
    prompt = f"Original question: {query}\n\nExpert answers:{expert_text}\nSynthesize a clear final answer:"
    return call_api(prompt, system="Combine expert answers into one coherent response.",
                    model=API_MODELS["synthesizer"], max_tokens=500)

print("Agents ready.")

Agents ready.


In [8]:
# ═══════════════════════════════════════════════════════════
# ADAPTIVE PIPELINE + BASELINES
# ═══════════════════════════════════════════════════════════

def run_adaptive(query, verbose=True):
    trace = QueryTrace(query=query)
    t0 = time.time()
    if verbose: print(f"\n{'═'*70}\n📝 QUERY: {query}\n{'═'*70}")

    q_type, reasoning, subs, cls_meta = classify_query(query)
    trace.classification = q_type; trace.reasoning = reasoning; trace.sub_questions = subs
    trace.models_used.append(API_MODELS["router"])
    trace.total_cost += cls_meta.get("cost",0); trace.total_tokens += cls_meta.get("total_tokens",0)
    if verbose: print(f"\n🔍 CLASSIFY [{API_MODELS['router']}] → {q_type.upper()} ({cls_meta.get('latency',0):.2f}s)")

    if q_type == "simple":
        answer, meta = call_api(f"Answer concisely: {query}", model=API_MODELS["router"], max_tokens=200)
        trace.final_answer = answer
        trace.total_cost += meta.get("cost",0); trace.total_tokens += meta.get("total_tokens",0)
        if verbose: print(f"⚡ DIRECT [{API_MODELS['router']}]: {answer} ({meta.get('latency',0):.2f}s, ${meta.get('cost',0):.6f})")
    else:
        if verbose:
            print(f"🔀 DECOMPOSED → {len(subs)} sub-questions:")
            for i,sq in enumerate(subs,1):
                model = 'biomistral-local' if sq.domain=='medical' else API_MODELS.get(sq.domain=='math' and 'math' or 'synthesizer', '?')
                print(f"   SQ{i} [{sq.domain}→{model}]: {sq.text}")

        t_par = time.time()
        trace.sub_questions = answer_parallel(subs)
        par_time = time.time() - t_par

        if verbose: print(f"\n🤖 AGENTS ({par_time:.2f}s):")
        for i,sq in enumerate(trace.sub_questions,1):
            trace.total_cost += sq.cost; trace.total_tokens += sq.tokens
            if sq.model_used not in trace.models_used: trace.models_used.append(sq.model_used)
            if verbose:
                a = sq.answer[:200]+'...' if len(sq.answer)>200 else sq.answer
                print(f"   SQ{i} [{sq.domain}|{sq.model_used}] ({sq.latency:.1f}s, ${sq.cost:.6f})")
                print(f"       {a}")

        answer, s_meta = synthesize(query, trace.sub_questions)
        trace.final_answer = answer
        trace.total_cost += s_meta.get("cost",0); trace.total_tokens += s_meta.get("total_tokens",0)
        trace.models_used.append(API_MODELS["synthesizer"])
        if verbose: print(f"\n🧩 SYNTHESIS [{API_MODELS['synthesizer']}]: {answer[:300]}")

    trace.total_time = time.time() - t0
    if verbose:
        print(f"\n{'─'*70}")
        print(f"⏱ {trace.total_time:.2f}s | 📊 {trace.total_tokens} tok | 💰 ${trace.total_cost:.6f}")
        print(f"🧠 Models: {trace.models_used}")
    return trace

def run_baseline_cheap(query):
    answer, meta = call_api(f"Answer thoroughly: {query}", model="llama-3.1-8b-instant", max_tokens=500)
    return {"answer": answer, **meta}

def run_baseline_expensive(query):
    answer, meta = call_api(f"Answer thoroughly: {query}", model="llama-3.3-70b-versatile", max_tokens=500)
    return {"answer": answer, **meta}

print("Pipeline ready.")

Pipeline ready.


In [9]:
# Quick test
print("Test simple:")
_ = run_adaptive("What is the capital of France?")
print("\nTest multi-hop:")
_ = run_adaptive("What is the BMI of a 1.75m, 85kg person and what health risks does that carry?")

Test simple:

══════════════════════════════════════════════════════════════════════
📝 QUERY: What is the capital of France?
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llama-3.1-8b-instant] → SIMPLE (0.27s)
⚡ DIRECT [llama-3.1-8b-instant]: The capital of France is Paris. (0.26s, $0.000003)

──────────────────────────────────────────────────────────────────────
⏱ 0.53s | 📊 241 tok | 💰 $0.000013
🧠 Models: ['llama-3.1-8b-instant']

Test multi-hop:

══════════════════════════════════════════════════════════════════════
📝 QUERY: What is the BMI of a 1.75m, 85kg person and what health risks does that carry?
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llama-3.1-8b-instant] → MULTI_HOP (0.35s)
🔀 DECOMPOSED → 3 sub-questions:
   SQ1 [general→llama-3.3-70b-versatile]: What is the formula to calculate BMI?
   SQ2 [general→llama-3.3-70b-versatile]: What is the BMI of a 1.75m, 85kg person?
   SQ3 [medical→biomistral-lo

In [10]:
DEMO_QUERIES = [
    {"q": "What is the capital of France?", "type": "simple"},
    {"q": "Who wrote Hamlet?", "type": "simple"},
    {"q": "What year did World War II end?", "type": "simple"},
    {"q": "A patient with Type 2 diabetes on metformin has kidney function decline to GFR 25. What alternatives and dosage adjustments are needed?", "type": "multi_hop"},
    {"q": "What is the mechanism by which ACE inhibitors reduce blood pressure, and how does this interact with the RAAS?", "type": "multi_hop"},
    {"q": "A hospital has 500 patients. Probability of disease is 0.03. What is the expected number and standard deviation (binomial)?", "type": "multi_hop"},
    {"q": "Solve the integral of (3x^2 + 2x - 5) from x=0 to x=4.", "type": "multi_hop"},
    {"q": "Drug half-life is 6h, patient takes 200mg at 8am. How much remains at 8pm? Why does this matter for toxicity?", "type": "multi_hop"},
    {"q": "Clinical trial: Drug A reduces mortality 15% vs placebo (p=0.03, n=1200). Calculate NNT and explain clinical significance.", "type": "multi_hop"},
    {"q": "What is the BMI of a 1.75m, 85kg person and what health risks does that BMI carry?", "type": "multi_hop"},

    # --- Medical & Clinical Reasoning ---
    {"q": "A patient presents with a TSH of 12 mIU/L and a free T4 of 0.7 ng/dL; provide the diagnosis and recommended starting dose of Levothyroxine for a 75kg adult.", "type": "multi_hop"},
    {"q": "Compare the mechanism of action of GLP-1 agonists versus DPP-4 inhibitors in the context of glucose-dependent insulin secretion.", "type": "multi_hop"},
    {"q": "What is the standard antibiotic prophylaxis for a patient with a penicillin allergy undergoing a prosthetic heart valve replacement?", "type": "multi_hop"},
    {"q": "Explain the 'triple whammy' effect on the kidneys when combining ACE inhibitors, NSAIDs, and diuretics.", "type": "multi_hop"},
    {"q": "Analyze the diagnostic criteria for Sepsis-3 and how the qSOFA score differs from the traditional SIRS criteria.", "type": "multi_hop"},
    {"q": "A patient on warfarin has an INR of 7.0 with no active bleeding. What is the protocol for Vitamin K administration and dose holding?", "type": "multi_hop"},
    {"q": "Describe the pathophysiology of Heparin-Induced Thrombocytopenia (HIT) Type II and the preferred alternative anticoagulants.", "type": "multi_hop"},
    {"q": "What are the ECG manifestations of hyperkalemia, and at what serum potassium level does the risk of sine wave patterns increase?", "type": "multi_hop"},
    {"q": "Differentiate between the presentation of Neuroleptic Malignant Syndrome and Serotonin Syndrome regarding neuromuscular findings.", "type": "multi_hop"},
    {"q": "Explain the role of the RANK/RANKL/OPG pathway in the development of osteoporosis and the mechanism of Denosumab.", "type": "multi_hop"},

    # --- Mathematics & Statistical Analysis ---
    {"q": "Calculate the volume of a solid of revolution formed by rotating y = sin(x) around the x-axis from x = 0 to x = pi.", "type": "multi_hop"},
    {"q": "Find the local maxima and minima for the function f(x) = 2x^3 - 9x^2 + 12x - 5 using the second derivative test.", "type": "multi_hop"},
    {"q": "In a Poisson distribution where lambda = 4.5, what is the probability of observing exactly 2 events?", "type": "multi_hop"},
    {"q": "Solve the differential equation dy/dx + 2y = e^-x with the initial condition y(0) = 1.", "type": "multi_hop"},
    {"q": "A bag contains 5 red, 3 blue, and 2 green marbles. If 3 marbles are drawn without replacement, what is the probability of getting at least one of each color?", "type": "multi_hop"},
    {"q": "Calculate the Eigenvalues for a 2x2 matrix where A = [[4, 1], [2, 3]].", "type": "multi_hop"},
    {"q": "Find the sum of the infinite geometric series: 15 + 5 + 5/3 + ...", "type": "multi_hop"},
    {"q": "Use the chain rule to find the derivative of g(x) = ln(cos^2(x)).", "type": "multi_hop"},
    {"q": "In a normal distribution with mu = 100 and sigma = 15, find the z-score corresponding to the 90th percentile.", "type": "multi_hop"},
    {"q": "Determine if the series sum_{n=1}^{inf} (-1)^n / (n^2 + 1) converges absolutely, conditionally, or diverges.", "type": "multi_hop"},

    # --- Cross-Domain: Med-Math & Pharmacokinetics ---
    {"q": "A drug follows first-order kinetics with a Vd of 40L and a clearance of 2 L/h. Calculate the elimination half-life and the steady-state concentration for a 100mg TID regimen.", "type": "multi_hop"},
    {"q": "Using the Cockcroft-Gault equation, calculate the CrCl for a 68-year-old female, 65kg, with a serum creatinine of 1.4 mg/dL.", "type": "multi_hop"},
    {"q": "A pediatric patient (15kg) requires a 5 mg/kg dose of a medication available as 25mg/5mL syrup. How many milliliters should be administered?", "type": "multi_hop"},
    {"q": "Calculate the Absolute Neutrophil Count (ANC) for a patient with a total WBC of 2,500, 30% Neutrophils, and 5% Bands.", "type": "multi_hop"},
    {"q": "A clinical study reports a Relative Risk (RR) of 0.65 for a new drug. If the baseline risk is 12%, calculate the NNT.", "type": "multi_hop"},
    {"q": "Determine the infusion rate in mL/hr to deliver 5 mcg/kg/min of dopamine to an 80kg patient using a concentration of 400mg/250mL.", "type": "multi_hop"},
    {"q": "A patient’s arterial blood gas shows pH 7.25, PaCO2 55, and HCO3 26. Calculate the expected compensation and identify the primary acid-base disorder.", "type": "multi_hop"},
    {"q": "Use the Parkland Formula to calculate the total fluid requirements in the first 24 hours for a 70kg patient with 30% Total Body Surface Area burns.", "type": "multi_hop"},
    {"q": "Calculate the corrected calcium for a patient with a total calcium of 7.5 mg/dL and an albumin of 2.2 g/dL.", "type": "multi_hop"},
    {"q": "If a test has 95% sensitivity and 90% specificity, calculate the Positive Predictive Value (PPV) in a population with a 5% prevalence.", "type": "multi_hop"},

    # --- Logic, Reasoning & Synthesis ---
    {"q": "Compare the ethical implications of using AI-driven diagnostic tools in rural vs. urban healthcare settings.", "type": "multi_hop"},
    {"q": "Synthesize a rebuttal to the argument that high-protein diets are universally damaging to renal function in healthy adults.", "type": "multi_hop"},
    {"q": "Explain how the 'Observer Effect' in physics might be used as a metaphor for Hawthorne Effect in clinical trials.", "type": "multi_hop"},
    {"q": "Evaluate the impact of the Flexner Report of 1910 on the current structure of North American medical education.", "type": "multi_hop"},
    {"q": "Create a logical framework to prioritize organ transplant recipients using both utilitarian and egalitarian principles.", "type": "multi_hop"},
    {"q": "Describe the cascade of events in a 'cytokine storm' and identify three logical points for therapeutic intervention.", "type": "multi_hop"},
    {"q": "Analyze why a drug might show high efficacy in Phase II trials but fail in Phase III, focusing on statistical power and population diversity.", "type": "multi_hop"},
    {"q": "Explain the logic behind 'Double-Blind' studies—how does it specifically mitigate both confirmation bias and the placebo effect?", "type": "multi_hop"},
    {"q": "Compare the economic viability of 'local GPU' inference versus 'API-based' inference for a hospital processing 10,000 queries daily.", "type": "multi_hop"},
    {"q": "Formulate a concise summary of how CRISPR-Cas9 technology differs logically from traditional gene therapy techniques.", "type": "multi_hop"},
]

print(f"{len(DEMO_QUERIES)} queries ready.")

50 queries ready.


In [9]:
# ═══════════════════════════════════════════════════════════
# FULL RUN: ADAPTIVE + CHEAP + EXPENSIVE
# ═══════════════════════════════════════════════════════════

res_adp, res_chp, res_exp = [], [], []

for i, demo in enumerate(DEMO_QUERIES, 1):
    print(f"\n{'▓'*70}")
    print(f"  [{i}/{len(DEMO_QUERIES)}] {demo['q'][:70]}")
    print(f"{'▓'*70}")

    trace = run_adaptive(demo["q"], verbose=True)
    res_adp.append({"query":demo["q"],"expected":demo["type"],"actual":trace.classification,
        "correct":demo["type"]==trace.classification,"answer":trace.final_answer,
        "time":trace.total_time,"tokens":trace.total_tokens,"cost":trace.total_cost,
        "models":trace.models_used})
    time.sleep(2)

    r = run_baseline_cheap(demo["q"])
    res_chp.append({"query":demo["q"],"answer":r.get("answer",""),"latency":r.get("latency",0),
        "total_tokens":r.get("total_tokens",0),"cost":r.get("cost",0)})
    time.sleep(1)

    r = run_baseline_expensive(demo["q"])
    res_exp.append({"query":demo["q"],"answer":r.get("answer",""),"latency":r.get("latency",0),
        "total_tokens":r.get("total_tokens",0),"cost":r.get("cost",0)})
    time.sleep(1)

print("\n✅ All runs complete!")


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  [1/50] What is the capital of France?
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

══════════════════════════════════════════════════════════════════════
📝 QUERY: What is the capital of France?
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llama-3.1-8b-instant] → SIMPLE (0.28s)
⚡ DIRECT [llama-3.1-8b-instant]: The capital of France is Paris. (0.13s, $0.000003)

──────────────────────────────────────────────────────────────────────
⏱ 0.41s | 📊 241 tok | 💰 $0.000013
🧠 Models: ['llama-3.1-8b-instant']

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  [2/50] Who wrote Hamlet?
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

══════════════════════════════════════════════════════════════════════
📝 QUERY: Who wrote Hamlet?
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llam

In [12]:
# ═══════════════════════════════════════════════════════════
# RELOAD FROM JSON (only if you restarted runtime)
# ═══════════════════════════════════════════════════════════
# Uncomment the lines below if reloading:

import json
with open("results_50q.json", "r") as f:
    save_data = json.load(f)
res_adp = save_data["adaptive"]
res_chp = save_data["cheap"]
res_exp = save_data["expensive"]
DEMO_QUERIES = save_data["queries"]
print(f"Loaded {len(res_adp)} results.")

Loaded 50 results.


In [13]:
# ═══════════════════════════════════════════════════════════
# RATE-LIMIT-SAFE SCORING WITH CHECKPOINT
# ═══════════════════════════════════════════════════════════
import time, json, re

JUDGE_SYS = """You are an impartial answer quality judge. Rate 1-10.
Consider: accuracy, completeness, specificity, clinical/mathematical correctness.
Respond ONLY with JSON: {"score": N, "reason": "brief"}"""

def judge(question, answer):
    if not answer or len(answer.strip()) < 5:
        return 1, "empty"
    prompt = f"Question: {question}\n\nAnswer: {answer[:1000]}\n\nRate 1-10:"
    raw, _ = call_api(prompt, system=JUDGE_SYS, model="llama-3.3-70b-versatile",
                      max_tokens=80, temperature=0)
    try:
        p = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
        return int(p.get("score", 5)), p.get("reason", "")
    except:
        return 5, "parse error"

# Load existing scores if any
try:
    with open("scores_checkpoint.json", "r") as f:
        scores = json.load(f)
    start_from = len(scores["adp"])
    print(f"Resuming from Q{start_from + 1} (already scored {start_from})")
except:
    scores = {"adp": [], "chp": [], "exp": []}
    start_from = 0
    print("Starting fresh.")

total = len(DEMO_QUERIES)
BATCH_SIZE = 5
COOLDOWN = 30  # seconds between batches

for i in range(start_from, total):
    q = DEMO_QUERIES[i]["q"] if isinstance(DEMO_QUERIES[i], dict) else DEMO_QUERIES[i]

    print(f"  Scoring Q{i+1}/{total}: {q[:55]}...")

    sa, ra = judge(q, res_adp[i]["answer"])
    time.sleep(3)
    sc, rc = judge(q, res_chp[i]["answer"])
    time.sleep(3)
    se, re_ = judge(q, res_exp[i]["answer"])
    time.sleep(3)

    scores["adp"].append(sa)
    scores["chp"].append(sc)
    scores["exp"].append(se)

    print(f"    Adaptive:{sa}/10 | 8B:{sc}/10 | 70B:{se}/10")

    # Checkpoint after every query
    with open("scores_checkpoint.json", "w") as f:
        json.dump(scores, f)

    # Cooldown every BATCH_SIZE queries
    if (i + 1) % BATCH_SIZE == 0 and i + 1 < total:
        print(f"  ⏸ Cooldown {COOLDOWN}s (scored {i+1}/{total})...")
        time.sleep(COOLDOWN)

print(f"\n✅ All {total} queries scored!")
print(f"   Saved to scores_checkpoint.json")

Starting fresh.
  Scoring Q1/50: What is the capital of France?...
    Adaptive:10/10 | 8B:10/10 | 70B:10/10
  Scoring Q2/50: Who wrote Hamlet?...
    Adaptive:10/10 | 8B:9/10 | 70B:9/10
  Scoring Q3/50: What year did World War II end?...
    Adaptive:10/10 | 8B:10/10 | 70B:10/10
  Scoring Q4/50: A patient with Type 2 diabetes on metformin has kidney ...
    Adaptive:9/10 | 8B:8/10 | 70B:9/10
  Scoring Q5/50: What is the mechanism by which ACE inhibitors reduce bl...
    Adaptive:9/10 | 8B:9/10 | 70B:8/10
  ⏸ Cooldown 30s (scored 5/50)...
  Scoring Q6/50: A hospital has 500 patients. Probability of disease is ...
    Adaptive:10/10 | 8B:10/10 | 70B:10/10
  Scoring Q7/50: Solve the integral of (3x^2 + 2x - 5) from x=0 to x=4....
    Adaptive:10/10 | 8B:10/10 | 70B:8/10
  Scoring Q8/50: Drug half-life is 6h, patient takes 200mg at 8am. How m...
    Adaptive:9/10 | 8B:9/10 | 70B:9/10
  Scoring Q9/50: Clinical trial: Drug A reduces mortality 15% vs placebo...
    Adaptive:8/10 | 8B:8/10 | 

In [13]:
# ═══════════════════════════════════════════════════════════
# SAVE ALL RESULTS TO JSON (no API calls needed)
# ═══════════════════════════════════════════════════════════
import json

save_data = {
    "adaptive": res_adp,
    "cheap": res_chp,
    "expensive": res_exp,
    "queries": DEMO_QUERIES,
}

with open("results_50q.json", "w") as f:
    json.dump(save_data, f, indent=2, default=str)

print(f"✅ Saved {len(res_adp)} adaptive + {len(res_chp)} cheap + {len(res_exp)} expensive results")
print(f"   File: results_50q.json")
print(f"   Download this file! You can reload it anytime.")

✅ Saved 50 adaptive + 50 cheap + 50 expensive results
   File: results_50q.json
   Download this file! You can reload it anytime.


In [14]:
# ═══════════════════════════════════════════════════════════
# FULL ANALYSIS WITH ALL 50 QUERIES
# ═══════════════════════════════════════════════════════════

total = len(DEMO_QUERIES)

print("\n" + "█" * 75)
print("  RESULTS: ADAPTIVE vs ALWAYS-CHEAP (8B) vs ALWAYS-EXPENSIVE (70B)")
print("  " + f"{total} queries evaluated")
print("█" * 75)

# Per-query table
print(f"\n{'#':<4s} {'Query':<48s} {'Adp':>3s} {'8B':>3s} {'70B':>3s}  {'A$':>8s} {'C$':>8s} {'E$':>8s}")
print(f"{'─'*4} {'─'*48} {'─'*3} {'─'*3} {'─'*3}  {'─'*8} {'─'*8} {'─'*8}")

for i in range(total):
    demo = DEMO_QUERIES[i]
    q_text = demo["q"] if isinstance(demo, dict) else demo
    q_type = demo.get("type", "?") if isinstance(demo, dict) else "?"
    q = q_text[:46] + '..'
    sa, sc, se = scores['adp'][i], scores['chp'][i], scores['exp'][i]
    ca = res_adp[i].get('cost', 0)
    cc = res_chp[i].get('cost', 0)
    ce = res_exp[i].get('cost', 0)
    print(f"{i+1:<4d} {q:<48s} {sa:>3d} {sc:>3d} {se:>3d}  ${ca:.5f} ${cc:.5f} ${ce:.5f}")

# Averages
avg_a = sum(scores['adp']) / total
avg_c = sum(scores['chp']) / total
avg_e = sum(scores['exp']) / total
tot_a = sum(r.get('cost', 0) for r in res_adp)
tot_c = sum(r.get('cost', 0) for r in res_chp)
tot_e = sum(r.get('cost', 0) for r in res_exp)

print(f"\n{'':4s} {'AVERAGE':<48s} {avg_a:>3.1f} {avg_c:>3.1f} {avg_e:>3.1f}  ${tot_a:.5f} ${tot_c:.5f} ${tot_e:.5f}")

# By type
print("\n" + "─" * 80)
print("BY QUERY TYPE:")
for qt in ['simple', 'multi_hop']:
    idx = [i for i in range(total) if isinstance(DEMO_QUERIES[i], dict) and DEMO_QUERIES[i].get('type') == qt]
    if not idx:
        continue
    a_s = [scores['adp'][i] for i in idx]
    c_s = [scores['chp'][i] for i in idx]
    e_s = [scores['exp'][i] for i in idx]
    a_c = sum(res_adp[i].get('cost', 0) for i in idx)
    c_c = sum(res_chp[i].get('cost', 0) for i in idx)
    e_c = sum(res_exp[i].get('cost', 0) for i in idx)
    print(f"\n  {qt.upper()} ({len(idx)} queries):")
    print(f"    Quality:  Adaptive {sum(a_s)/len(a_s):.1f}/10 | Cheap 8B {sum(c_s)/len(c_s):.1f}/10 | Expensive 70B {sum(e_s)/len(e_s):.1f}/10")
    print(f"    Cost:     Adaptive ${a_c:.5f} | Cheap ${c_c:.5f} | Expensive ${e_c:.5f}")

# Win/loss/tie counts
print("\n" + "─" * 80)
print("HEAD-TO-HEAD (Adaptive vs each baseline):")
for name, other_scores in [("Cheap 8B", scores['chp']), ("Expensive 70B", scores['exp'])]:
    wins = sum(1 for a, b in zip(scores['adp'], other_scores) if a > b)
    ties = sum(1 for a, b in zip(scores['adp'], other_scores) if a == b)
    losses = sum(1 for a, b in zip(scores['adp'], other_scores) if a < b)
    print(f"  vs {name:15s}: W={wins:2d} | T={ties:2d} | L={losses:2d}")

# Routing accuracy
correct = sum(1 for r in res_adp if r.get('correct', False))
print(f"\n📊 Routing Accuracy: {correct}/{total} ({correct/total:.0%})")

# Model usage
print("\n🧠 Model Usage Across All Queries:")
model_counts = {}
for r in res_adp:
    for m in r.get('models', []):
        model_counts[m] = model_counts.get(m, 0) + 1
for m, c in sorted(model_counts.items(), key=lambda x: -x[1]):
    bar = '█' * c + '░' * (total - c)
    print(f"  {m:35s}: {bar} {c}/{total}")

# The pitch
print("\n" + "█" * 75)
print("  THE PITCH")
print("█" * 75)
cost_save_vs_exp = (1 - tot_a / tot_e) * 100 if tot_e > 0 else 0
print(f"\n  🎯 Avg Quality:  Adaptive {avg_a:.1f} | Cheap 8B {avg_c:.1f} | Expensive 70B {avg_e:.1f}")
print(f"  💰 Total Cost:   Adaptive ${tot_a:.5f} | Cheap ${tot_c:.5f} | Expensive ${tot_e:.5f}")
print(f"  💰 vs Expensive: {cost_save_vs_exp:+.1f}%")
print(f"  📊 Routing:      {correct}/{total} correct")
print(f"\n  Architecture: 4 models, 3 orgs, hybrid local+API")
print(f"    • BioMistral-7B (EPFL) — LOCAL, PubMed-trained")
print(f"    • Qwen3-32B (Alibaba) — API, reasoning model")
print(f"    • LLaMA 3.1 8B (Meta) — API, fast router")
print(f"    • LLaMA 3.3 70B (Meta) — API, synthesizer")


███████████████████████████████████████████████████████████████████████████
  RESULTS: ADAPTIVE vs ALWAYS-CHEAP (8B) vs ALWAYS-EXPENSIVE (70B)
  50 queries evaluated
███████████████████████████████████████████████████████████████████████████

#    Query                                            Adp  8B 70B        A$       C$       E$
──── ──────────────────────────────────────────────── ─── ─── ───  ──────── ──────── ────────
1    What is the capital of France?..                  10  10  10  $0.00001 $0.00003 $0.00032
2    Who wrote Hamlet?..                               10   9   9  $0.00001 $0.00004 $0.00043
3    What year did World War II end?..                 10  10  10  $0.00002 $0.00002 $0.00013
4    A patient with Type 2 diabetes on metformin ha..   9   8   9  $0.00072 $0.00004 $0.00049
5    What is the mechanism by which ACE inhibitors ..   9   9   8  $0.00053 $0.00004 $0.00048
6    A hospital has 500 patients. Probability of di..  10  10  10  $0.00175 $0.00003 $0.00031
7   

In [15]:
# ═══════════════════════════════════════════════════════════
# SIDE-BY-SIDE ANSWERS — Show the best examples
# Where adaptive scored HIGHER than both baselines
# ═══════════════════════════════════════════════════════════

print("\n" + "█" * 75)
print("  BEST SHOWCASE EXAMPLES (Adaptive beats BOTH baselines)")
print("█" * 75)

showcased = 0
for i in range(total):
    sa, sc, se = scores['adp'][i], scores['chp'][i], scores['exp'][i]
    if sa > sc and sa > se:  # Adaptive wins both
        showcased += 1
        demo = DEMO_QUERIES[i]
        q = demo["q"] if isinstance(demo, dict) else demo
        print(f"\n{'━' * 75}")
        print(f"Q{i+1}: {q}")
        print(f"Scores: Adaptive {sa}/10 ⭐ | 8B {sc}/10 | 70B {se}/10")
        print(f"Models: {res_adp[i].get('models', [])}")
        print(f"\n┌─ ADAPTIVE (${res_adp[i].get('cost',0):.5f})")
        print(f"│ {res_adp[i]['answer'][:500]}{'...' if len(res_adp[i]['answer'])>500 else ''}")
        print(f"├─ CHEAP 8B")
        print(f"│ {res_chp[i]['answer'][:500]}{'...' if len(res_chp[i]['answer'])>500 else ''}")
        print(f"├─ EXPENSIVE 70B")
        print(f"│ {res_exp[i]['answer'][:500]}{'...' if len(res_exp[i]['answer'])>500 else ''}")
        print(f"└─")

if showcased == 0:
    print("\n  No queries where adaptive beat both. Showing where it beat at least one:")
    for i in range(total):
        sa, sc, se = scores['adp'][i], scores['chp'][i], scores['exp'][i]
        if sa > sc or sa > se:
            demo = DEMO_QUERIES[i]
            q = demo["q"] if isinstance(demo, dict) else demo
            print(f"  Q{i+1}: {q[:60]}... → Adp:{sa} | 8B:{sc} | 70B:{se}")

print(f"\n  Total showcase examples: {showcased}/{total}")


███████████████████████████████████████████████████████████████████████████
  BEST SHOWCASE EXAMPLES (Adaptive beats BOTH baselines)
███████████████████████████████████████████████████████████████████████████

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Q2: Who wrote Hamlet?
Scores: Adaptive 10/10 ⭐ | 8B 9/10 | 70B 9/10
Models: ['llama-3.1-8b-instant']

┌─ ADAPTIVE ($0.00001)
│ William Shakespeare wrote Hamlet.
├─ CHEAP 8B
│ Hamlet is a tragedy written by the renowned English playwright William Shakespeare. It is believed to have been written between 1599 and 1602, during the Elizabethan era. The exact date of its composition is not known, but it is thought to have been performed at the Globe Theatre in London around 1603.

William Shakespeare was an English playwright, poet, and actor who is widely regarded as one of the greatest writers in the English language. He wrote at least 38 plays and 154 sonnets, and his w...
├─ EXPENSIVE 70B
│ The authorship 

In [16]:
# ═══════════════════════════════════════════════════════════
# SAVE EVERYTHING FOR YOUR PRESENTATION
# ═══════════════════════════════════════════════════════════

final_report = {
    "summary": {
        "total_queries": total,
        "routing_accuracy": f"{correct}/{total}",
        "avg_quality_adaptive": avg_a,
        "avg_quality_cheap": avg_c,
        "avg_quality_expensive": avg_e,
        "total_cost_adaptive": tot_a,
        "total_cost_cheap": tot_c,
        "total_cost_expensive": tot_e,
    },
    "scores": scores,
    "results_adaptive": res_adp,
    "results_cheap": res_chp,
    "results_expensive": res_exp,
    "queries": DEMO_QUERIES,
}

with open("final_report_50q.json", "w") as f:
    json.dump(final_report, f, indent=2, default=str)

print("✅ Full report saved to final_report_50q.json")
print("   Download this for your presentation!")

✅ Full report saved to final_report_50q.json
   Download this for your presentation!


In [26]:
# ═══════════════════════════════════════════════════════════
# TRY YOUR OWN — Run + Analyse in one cell
# ═══════════════════════════════════════════════════════════

my_q = "A clinical study reports a Relative Risk (RR) of 0.65 for a new drug. If the baseline risk is 12%, calculate the NNT."

# Run all three
print("=" * 70)
print("ADAPTIVE:")
print("=" * 70)
trace = run_adaptive(my_q)
adp_answer = trace.final_answer
adp_cost = trace.total_cost
adp_time = trace.total_time
adp_models = trace.models_used

time.sleep(3)

print("\n" + "=" * 70)
print("CHEAP 8B:")
print("=" * 70)
r_chp = run_baseline_cheap(my_q)
chp_answer = r_chp.get("answer", "")
chp_cost = r_chp.get("cost", 0)
chp_time = r_chp.get("latency", 0)
print(f"  {chp_answer[:500]}")

time.sleep(3)

print("\n" + "=" * 70)
print("EXPENSIVE 70B:")
print("=" * 70)
r_exp = run_baseline_expensive(my_q)
exp_answer = r_exp.get("answer", "")
exp_cost = r_exp.get("cost", 0)
exp_time = r_exp.get("latency", 0)
print(f"  {exp_answer[:500]}")

time.sleep(3)

# ── Judge all three ──
print("\n" + "█" * 70)
print("  LLM JUDGE EVALUATION")
print("█" * 70)

score_adp, reason_adp = judge(my_q, adp_answer)
time.sleep(3)
score_chp, reason_chp = judge(my_q, chp_answer)
time.sleep(3)
score_exp, reason_exp = judge(my_q, exp_answer)

# ── Comparison table ──
print(f"\n{'System':<17s} {'Score':<8s} {'Time':<8s} {'Cost':<11s} {'Models':<40s} Reason")
print("─" * 130)
print(f"{'ADAPTIVE':<17s} {score_adp:<8d} {adp_time:<8.2f} ${adp_cost:<10.5f} {str(adp_models):<40s} {reason_adp}")
print(f"{'CHEAP (8B)':<17s} {score_chp:<8d} {chp_time:<8.2f} ${chp_cost:<10.5f} {'llama-3.1-8b-instant':<40s} {reason_chp}")
print(f"{'EXPENSIVE (70B)':<17s} {score_exp:<8d} {exp_time:<8.2f} ${exp_cost:<10.5f} {'llama-3.3-70b-versatile':<40s} {reason_exp}")

# ── Verdict ──
print("\n" + "━" * 70)
best = max([(score_adp, "Adaptive"), (score_chp, "Cheap 8B"), (score_exp, "Expensive 70B")])
print(f"🏆 Best quality: {best[1]} ({best[0]}/10)")

if score_adp >= score_exp and score_adp >= score_chp:
    print(f"✅ Adaptive WINS with best quality!")
    if adp_cost < exp_cost:
        print(f"   AND costs {(1-adp_cost/exp_cost)*100:.0f}% less than expensive baseline.")
    elif adp_cost > 0:
        print(f"   (costs ${adp_cost:.5f} vs cheap ${chp_cost:.5f} / expensive ${exp_cost:.5f})")
elif score_adp >= score_exp:
    print(f"✅ Adaptive matches/beats 70B ({score_adp} vs {score_exp}), beats 8B ({score_adp} vs {score_chp})")
elif score_adp > score_chp:
    print(f"⚠️ Adaptive beats 8B ({score_adp} vs {score_chp}) but 70B wins ({score_exp})")
else:
    print(f"💡 Baselines outperformed on this query. Adaptive: {score_adp}, 8B: {score_chp}, 70B: {score_exp}")
print("━" * 70)

ADAPTIVE:

══════════════════════════════════════════════════════════════════════
📝 QUERY: A clinical study reports a Relative Risk (RR) of 0.65 for a new drug. If the baseline risk is 12%, calculate the NNT.
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llama-3.1-8b-instant] → MULTI_HOP (0.33s)
🔀 DECOMPOSED → 3 sub-questions:
   SQ1 [general→llama-3.3-70b-versatile]: What is the formula to calculate the Number Needed to Treat (NNT)?
   SQ2 [general→llama-3.3-70b-versatile]: How is the Relative Risk (RR) related to the Odds Ratio (OR)?
   SQ3 [general→llama-3.3-70b-versatile]: Given a baseline risk of 12%, what is the absolute risk reduction (ARR)?

🤖 AGENTS (1.03s):
   SQ1 [general|llama-3.3-70b-versatile] (0.7s, $0.000212)
       The formula to calculate the Number Needed to Treat (NNT) is:

NNT = 1 / ARR

Where:
- NNT = Number Needed to Treat
- ARR = Absolute Risk Reduction

The Absolute Risk Reduction (ARR) is calculated as:...
   SQ2 [general|

In [27]:
# ═══════════════════════════════════════════════════════════
# TRY YOUR OWN — Run + Analyse in one cell
# ═══════════════════════════════════════════════════════════

my_q = "Explain the differential metabolic risk for a patient on Simvastatin who is a CYP2C93 carrier when co-administered with Warfarin, versus a non-carrier."

# Run all three
print("=" * 70)
print("ADAPTIVE:")
print("=" * 70)
trace = run_adaptive(my_q)
adp_answer = trace.final_answer
adp_cost = trace.total_cost
adp_time = trace.total_time
adp_models = trace.models_used

time.sleep(3)

print("\n" + "=" * 70)
print("CHEAP 8B:")
print("=" * 70)
r_chp = run_baseline_cheap(my_q)
chp_answer = r_chp.get("answer", "")
chp_cost = r_chp.get("cost", 0)
chp_time = r_chp.get("latency", 0)
print(f"  {chp_answer[:500]}")

time.sleep(3)

print("\n" + "=" * 70)
print("EXPENSIVE 70B:")
print("=" * 70)
r_exp = run_baseline_expensive(my_q)
exp_answer = r_exp.get("answer", "")
exp_cost = r_exp.get("cost", 0)
exp_time = r_exp.get("latency", 0)
print(f"  {exp_answer[:500]}")

time.sleep(3)

# ── Judge all three ──
print("\n" + "█" * 70)
print("  LLM JUDGE EVALUATION")
print("█" * 70)

score_adp, reason_adp = judge(my_q, adp_answer)
time.sleep(3)
score_chp, reason_chp = judge(my_q, chp_answer)
time.sleep(3)
score_exp, reason_exp = judge(my_q, exp_answer)

# ── Comparison table ──
print(f"\n{'System':<17s} {'Score':<8s} {'Time':<8s} {'Cost':<11s} {'Models':<40s} Reason")
print("─" * 130)
print(f"{'ADAPTIVE':<17s} {score_adp:<8d} {adp_time:<8.2f} ${adp_cost:<10.5f} {str(adp_models):<40s} {reason_adp}")
print(f"{'CHEAP (8B)':<17s} {score_chp:<8d} {chp_time:<8.2f} ${chp_cost:<10.5f} {'llama-3.1-8b-instant':<40s} {reason_chp}")
print(f"{'EXPENSIVE (70B)':<17s} {score_exp:<8d} {exp_time:<8.2f} ${exp_cost:<10.5f} {'llama-3.3-70b-versatile':<40s} {reason_exp}")

# ── Verdict ──
print("\n" + "━" * 70)
best = max([(score_adp, "Adaptive"), (score_chp, "Cheap 8B"), (score_exp, "Expensive 70B")])
print(f"🏆 Best quality: {best[1]} ({best[0]}/10)")

if score_adp >= score_exp and score_adp >= score_chp:
    print(f"✅ Adaptive WINS with best quality!")
    if adp_cost < exp_cost:
        print(f"   AND costs {(1-adp_cost/exp_cost)*100:.0f}% less than expensive baseline.")
    elif adp_cost > 0:
        print(f"   (costs ${adp_cost:.5f} vs cheap ${chp_cost:.5f} / expensive ${exp_cost:.5f})")
elif score_adp >= score_exp:
    print(f"✅ Adaptive matches/beats 70B ({score_adp} vs {score_exp}), beats 8B ({score_adp} vs {score_chp})")
elif score_adp > score_chp:
    print(f"⚠️ Adaptive beats 8B ({score_adp} vs {score_chp}) but 70B wins ({score_exp})")
else:
    print(f"💡 Baselines outperformed on this query. Adaptive: {score_adp}, 8B: {score_chp}, 70B: {score_exp}")
print("━" * 70)

ADAPTIVE:

══════════════════════════════════════════════════════════════════════
📝 QUERY: Explain the differential metabolic risk for a patient on Simvastatin who is a CYP2C93 carrier when co-administered with Warfarin, versus a non-carrier.
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llama-3.1-8b-instant] → MULTI_HOP (0.34s)
🔀 DECOMPOSED → 3 sub-questions:
   SQ1 [medical→biomistral-local]: What is the role of CYP2C9 in simvastatin metabolism?
   SQ2 [medical→biomistral-local]: How does CYP2C9 genetic variation affect warfarin metabolism?
   SQ3 [medical→biomistral-local]: What is the potential interaction between simvastatin and warfarin in CYP2C9 carriers?

🤖 AGENTS (27.80s):
   SQ1 [medical|biomistral-local] (10.4s, $0.000000)
       Simvastatin is metabolized by CYP3A4 and CYP2C9 to form 6-hydroxymethyl simvastatin and 6-hydroxy simvastatin, respectively. The former is the major metabolic pathway. The CYP2C9-mediated pathway is m...
   SQ2 

In [29]:
# ═══════════════════════════════════════════════════════════
# TRY YOUR OWN — Run + Analyse in one cell
# ═══════════════════════════════════════════════════════════

my_q = "Find the local maxima and minima for the function f(x) = 2x^3 - 9x^2 + 12x - 5 using the second derivative test."

# Run all three
print("=" * 70)
print("ADAPTIVE:")
print("=" * 70)
trace = run_adaptive(my_q)
adp_answer = trace.final_answer
adp_cost = trace.total_cost
adp_time = trace.total_time
adp_models = trace.models_used

time.sleep(3)

print("\n" + "=" * 70)
print("CHEAP 8B:")
print("=" * 70)
r_chp = run_baseline_cheap(my_q)
chp_answer = r_chp.get("answer", "")
chp_cost = r_chp.get("cost", 0)
chp_time = r_chp.get("latency", 0)
print(f"  {chp_answer[:500]}")

time.sleep(3)

print("\n" + "=" * 70)
print("EXPENSIVE 70B:")
print("=" * 70)
r_exp = run_baseline_expensive(my_q)
exp_answer = r_exp.get("answer", "")
exp_cost = r_exp.get("cost", 0)
exp_time = r_exp.get("latency", 0)
print(f"  {exp_answer[:500]}")

time.sleep(3)

# ── Judge all three ──
print("\n" + "█" * 70)
print("  LLM JUDGE EVALUATION")
print("█" * 70)

score_adp, reason_adp = judge(my_q, adp_answer)
time.sleep(3)
score_chp, reason_chp = judge(my_q, chp_answer)
time.sleep(3)
score_exp, reason_exp = judge(my_q, exp_answer)

# ── Comparison table ──
print(f"\n{'System':<17s} {'Score':<8s} {'Time':<8s} {'Cost':<11s} {'Models':<40s} Reason")
print("─" * 130)
print(f"{'ADAPTIVE':<17s} {score_adp:<8d} {adp_time:<8.2f} ${adp_cost:<10.5f} {str(adp_models):<40s} {reason_adp}")
print(f"{'CHEAP (8B)':<17s} {score_chp:<8d} {chp_time:<8.2f} ${chp_cost:<10.5f} {'llama-3.1-8b-instant':<40s} {reason_chp}")
print(f"{'EXPENSIVE (70B)':<17s} {score_exp:<8d} {exp_time:<8.2f} ${exp_cost:<10.5f} {'llama-3.3-70b-versatile':<40s} {reason_exp}")

# ── Verdict ──
print("\n" + "━" * 70)
best = max([(score_adp, "Adaptive"), (score_chp, "Cheap 8B"), (score_exp, "Expensive 70B")])
print(f"🏆 Best quality: {best[1]} ({best[0]}/10)")

if score_adp >= score_exp and score_adp >= score_chp:
    print(f"✅ Adaptive WINS with best quality!")
    if adp_cost < exp_cost:
        print(f"   AND costs {(1-adp_cost/exp_cost)*100:.0f}% less than expensive baseline.")
    elif adp_cost > 0:
        print(f"   (costs ${adp_cost:.5f} vs cheap ${chp_cost:.5f} / expensive ${exp_cost:.5f})")
elif score_adp >= score_exp:
    print(f"✅ Adaptive matches/beats 70B ({score_adp} vs {score_exp}), beats 8B ({score_adp} vs {score_chp})")
elif score_adp > score_chp:
    print(f"⚠️ Adaptive beats 8B ({score_adp} vs {score_chp}) but 70B wins ({score_exp})")
else:
    print(f"💡 Baselines outperformed on this query. Adaptive: {score_adp}, 8B: {score_chp}, 70B: {score_exp}")
print("━" * 70)

ADAPTIVE:

══════════════════════════════════════════════════════════════════════
📝 QUERY: Find the local maxima and minima for the function f(x) = 2x^3 - 9x^2 + 12x - 5 using the second derivative test.
══════════════════════════════════════════════════════════════════════

🔍 CLASSIFY [llama-3.1-8b-instant] → MULTI_HOP (0.44s)
🔀 DECOMPOSED → 3 sub-questions:
   SQ1 [math→qwen/qwen3-32b]: What is the first derivative of the function f(x) = 2x^3 - 9x^2 + 12x - 5?
   SQ2 [math→qwen/qwen3-32b]: How do you find the critical points using the first derivative?
   SQ3 [math→qwen/qwen3-32b]: What is the second derivative of the function f(x) = 2x^3 - 9x^2 + 12x - 5?

🤖 AGENTS (1.03s):
   SQ1 [math|qwen/qwen3-32b] (0.9s, $0.000252)
       <think>
Okay, so I need to find the first derivative of the function f(x) = 2x³ - 9x² + 12x - 5. Hmm, derivatives. Let me remember how to do this. I think the derivative of a function gives you the sl...
   SQ2 [math|qwen/qwen3-32b] (1.0s, $0.000248)
       <t

In [25]:
# 1. Fetch the last results from the execution lists
final_adp = res_adp[-1]
final_chp = res_chp[-1]
final_exp = res_exp[-1]
query_text = final_adp['query']

print(f"🔍 Evaluating Final Query: {query_text}\n")

# 2. Run the Judge on all three versions
score_adp, reason_adp = judge(query_text, final_adp['answer'])
score_chp, reason_chp = judge(query_text, final_chp['answer'])
score_exp, reason_exp = judge(query_text, final_exp['answer'])

# 3. Display Comparison Table
print(f"{'System':<15} | {'Score':<5} | {'Cost':<10} | {'Reasoning'}")
print("-" * 80)
print(f"{'ADAPTIVE':<15} | {score_adp:<5}/10 | ${final_adp['cost']:.5f}  | {reason_adp}")
print(f"{'CHEAP (8B)':<15} | {score_chp:<5}/10 | ${final_chp['cost']:.5f}  | {reason_chp}")
print(f"{'EXPENSIVE (70B)':<15} | {score_exp:<5}/10 | ${final_exp['cost']:.5f}  | {reason_exp}")

# 4. Summary Pitch for this specific query
print("\n" + "═"*80)
if score_adp >= score_exp:
    savings = (1 - (final_adp['cost'] / final_exp['cost'])) * 100
    print(f"🎯 SUCCESS: Adaptive match/beat 70B quality while saving {savings:.1f}% in cost.")
else:
    print(f"💡 NOTE: The 70B baseline outperformed the Adaptive route by {score_exp - score_adp} points.")
print("═"*80)

🔍 Evaluating Final Query: Formulate a concise summary of how CRISPR-Cas9 technology differs logically from traditional gene therapy techniques.

System          | Score | Cost       | Reasoning
--------------------------------------------------------------------------------
ADAPTIVE        | 9    /10 | $0.00143  | accurate and comprehensive summary of CRISPR-Cas9 vs traditional gene therapy
CHEAP (8B)      | 9    /10 | $0.00004  | accurate and specific comparison of CRISPR-Cas9 and traditional gene therapy
EXPENSIVE (70B) | 9    /10 | $0.00048  | Accurate and comprehensive summary, but lacks a concluding statement

════════════════════════════════════════════════════════════════════════════════
🎯 SUCCESS: Adaptive match/beat 70B quality while saving -197.4% in cost.
════════════════════════════════════════════════════════════════════════════════
